# Brier Score Backtest

For every match in `tennis.db`, use the **YTD stats recorded at that match** as the Bayesian prior,
run the Monte Carlo simulator, and score the resulting win probability against the actual outcome.

Brier score = mean((p_predicted - outcome)^2), lower is better, 0.25 = random.

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

In [2]:
# ── Simulation engine (copied from monte_carlo_basic.ipynb) ───────────────────

class BetaModel:
    def __init__(self, career_rate, prior_strength, lam=0.95, warmup=None):
        self.alpha_prior = career_rate * prior_strength
        self.beta_prior  = (1.0 - career_rate) * prior_strength
        self.alpha_match = 0.0
        self.beta_match  = 0.0
        self.lam    = lam
        self.warmup = warmup if warmup is not None else int(1 / (1 - lam))
        self.n_obs  = 0

    def get_p(self):
        if self.n_obs < self.warmup:
            return self.alpha_prior / (self.alpha_prior + self.beta_prior)
        total = (self.alpha_prior + self.beta_prior +
                 self.alpha_match + self.beta_match)
        return (self.alpha_prior + self.alpha_match) / total

    def update(self, won):
        self.alpha_match *= self.lam
        self.beta_match  *= self.lam
        if won:
            self.alpha_match += 1.0
        else:
            self.beta_match  += 1.0
        self.n_obs += 1

    def reset(self):
        self.alpha_match = 0.0
        self.beta_match  = 0.0
        self.n_obs       = 0


class Player:
    def __init__(self, name, stats, prior_strength=200, lam=0.95):
        self.name  = name
        self.stats = stats
        self.serve_first_model   = BetaModel(stats['win_first'],    prior_strength, lam)
        self.serve_second_model  = BetaModel(stats['win_second'],   prior_strength, lam)
        self.return_first_model  = BetaModel(stats['return_first'], prior_strength, lam)
        self.return_second_model = BetaModel(stats['return_second'],prior_strength, lam)
        self.bp_save_model = BetaModel(
            career_rate=stats['bp_save_rate'], prior_strength=stats['bp_save_faced'],
            lam=lam, warmup=20)
        self.bp_convert_model = BetaModel(
            career_rate=stats['bp_convert_rate'], prior_strength=stats['bp_convert_opps'],
            lam=lam, warmup=20)

    def reset(self):
        for m in [self.serve_first_model, self.serve_second_model,
                  self.return_first_model, self.return_second_model,
                  self.bp_save_model, self.bp_convert_model]:
            m.reset()


def is_break_point(server_pts, receiver_pts):
    if receiver_pts == 3 and server_pts < 3:
        return True
    if receiver_pts >= 4 and server_pts >= 3 and receiver_pts == server_pts + 1:
        return True
    return False


def sim_point(server, receiver, server_pts, receiver_pts):
    bp = is_break_point(server_pts, receiver_pts)
    first_in = server.stats['first_in']
    if bp:
        p_win = (server.bp_save_model.get_p() + (1.0 - receiver.bp_convert_model.get_p())) / 2.0
        server_won = np.random.random() < p_win
        server.bp_save_model.update(server_won)
        receiver.bp_convert_model.update(not server_won)
        if np.random.random() < first_in:
            server.serve_first_model.update(server_won)
            receiver.return_first_model.update(not server_won)
        else:
            server.serve_second_model.update(server_won)
            receiver.return_second_model.update(not server_won)
    else:
        p_win_1st = (server.serve_first_model.get_p() + (1.0 - receiver.return_first_model.get_p())) / 2.0
        p_win_2nd = (server.serve_second_model.get_p() + (1.0 - receiver.return_second_model.get_p())) / 2.0
        if np.random.random() < first_in:
            server_won = np.random.random() < p_win_1st
            server.serve_first_model.update(server_won)
            receiver.return_first_model.update(not server_won)
        else:
            server_won = np.random.random() < p_win_2nd
            server.serve_second_model.update(server_won)
            receiver.return_second_model.update(not server_won)
    return server_won


def sim_game(server, receiver):
    score = [0, 0]
    while True:
        if sim_point(server, receiver, score[0], score[1]):
            score[0] += 1
        else:
            score[1] += 1
        if score[0] >= 4 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 4 and score[1] - score[0] >= 2:
            return False


def sim_tiebreak(p1, p2, p1_serves_first):
    score = [0, 0]
    point_count = 0
    while True:
        if point_count == 0:
            p1_serves = p1_serves_first
        else:
            p1_serves = p1_serves_first == (point_count % 2 == 0)
        server, receiver = (p1, p2) if p1_serves else (p2, p1)
        server_won = sim_point(server, receiver, 0, 0)
        p1_won = server_won if p1_serves else not server_won
        score[0 if p1_won else 1] += 1
        point_count += 1
        if score[0] >= 7 and score[0] - score[1] >= 2:
            return True
        if score[1] >= 7 and score[1] - score[0] >= 2:
            return False


def sim_set(p1, p2, p1_serves_first):
    games = [0, 0]
    p1_serving = p1_serves_first
    while True:
        server, receiver = (p1, p2) if p1_serving else (p2, p1)
        server_won = sim_game(server, receiver)
        p1_won_game = server_won if p1_serving else not server_won
        games[0 if p1_won_game else 1] += 1
        p1_serving = not p1_serving
        if games[0] == 6 and games[1] == 6:
            p1_won_tb = sim_tiebreak(p1, p2, p1_serving)
            games[0 if p1_won_tb else 1] += 1
            return games[0] > games[1], p1_serving
        if games[0] >= 6 and games[0] - games[1] >= 2:
            return True, p1_serving
        if games[1] >= 6 and games[1] - games[0] >= 2:
            return False, p1_serving


def sim_match(p1, p2, p1_serves_first=True, best_of=3):
    p1.reset()
    p2.reset()
    sets_needed = best_of // 2 + 1
    sets = [0, 0]
    p1_serving = p1_serves_first
    while sets[0] < sets_needed and sets[1] < sets_needed:
        p1_won_set, p1_serving = sim_set(p1, p2, p1_serving)
        sets[0 if p1_won_set else 1] += 1
    return sets[0] > sets[1]


def run_simulation(p1, p2, N=5_000, best_of=3):
    wins = sum(sim_match(p1, p2, best_of=best_of) for _ in range(N))
    return wins / N

In [3]:
# ── Load matches + YTD stats from DB ─────────────────────────────────────────

DB_PATH = 'tennis.db'
N_SIMS  = 5_000    # per match; gives ~±0.7% accuracy
PRIOR_STRENGTH = 200
LAM = 0.95

conn = sqlite3.connect(DB_PATH)

query = """
SELECT
    m.id            AS match_id,
    m.winner_player_id,
    m.number_of_sets,
    t.event_year,
    t.tournament_name,
    mp.player_id    AS p1_id,
    -- P1 YTD stats
    y1.first_serve_pct        AS p1_first_in,
    y1.first_serve_won_pct    AS p1_win_first,
    y1.second_serve_won_pct   AS p1_win_second,
    y1.first_return_won_pct   AS p1_return_first,
    y1.second_return_won_pct  AS p1_return_second,
    y1.bp_saved_pct           AS p1_bp_save_rate,
    y1.bp_faced               AS p1_bp_save_faced,
    y1.bp_converted_pct       AS p1_bp_convert_rate,
    y1.bp_opportunities       AS p1_bp_convert_opps,
    -- P2
    mp2.player_id   AS p2_id,
    -- P2 YTD stats
    y2.first_serve_pct        AS p2_first_in,
    y2.first_serve_won_pct    AS p2_win_first,
    y2.second_serve_won_pct   AS p2_win_second,
    y2.first_return_won_pct   AS p2_return_first,
    y2.second_return_won_pct  AS p2_return_second,
    y2.bp_saved_pct           AS p2_bp_save_rate,
    y2.bp_faced               AS p2_bp_save_faced,
    y2.bp_converted_pct       AS p2_bp_convert_rate,
    y2.bp_opportunities       AS p2_bp_convert_opps
FROM matches m
JOIN tournaments t ON t.id = m.tournament_id
JOIN match_players mp  ON mp.match_id  = m.id AND mp.is_player1 = 1
JOIN match_players mp2 ON mp2.match_id = m.id AND mp2.is_player1 = 0
JOIN match_ytd_stats y1 ON y1.match_id = m.id AND y1.player_id = mp.player_id
JOIN match_ytd_stats y2 ON y2.match_id = m.id AND y2.player_id = mp2.player_id
WHERE m.winner_player_id IS NOT NULL
"""

df = pd.read_sql_query(query, conn)
conn.close()

print(f'Matches loaded: {len(df):,}')
print(df[['event_year','tournament_name','p1_id','p2_id']].head(3))

Matches loaded: 10,363
   event_year           tournament_name p1_id p2_id
0        2023  Adelaide International 1  d643  mm58
1        2023  Adelaide International 1  d643  su55
2        2023  Adelaide International 1  mm58  ke29


In [4]:
# ── Drop rows with any null YTD stat ─────────────────────────────────────────

stat_cols = [
    'p1_first_in','p1_win_first','p1_win_second','p1_return_first','p1_return_second',
    'p1_bp_save_rate','p1_bp_save_faced','p1_bp_convert_rate','p1_bp_convert_opps',
    'p2_first_in','p2_win_first','p2_win_second','p2_return_first','p2_return_second',
    'p2_bp_save_rate','p2_bp_save_faced','p2_bp_convert_rate','p2_bp_convert_opps',
]

before = len(df)
df = df.dropna(subset=stat_cols)
print(f'Dropped {before - len(df):,} rows with null YTD stats, {len(df):,} remain')

Dropped 151 rows with null YTD stats, 10,212 remain


In [5]:
# ── Run simulations ───────────────────────────────────────────────────────────

def make_stats(row, prefix):
    return {
        'first_in':        row[f'{prefix}first_in']        / 100,
        'win_first':       row[f'{prefix}win_first']       / 100,
        'win_second':      row[f'{prefix}win_second']      / 100,
        'return_first':    row[f'{prefix}return_first']    / 100,
        'return_second':   row[f'{prefix}return_second']   / 100,
        'bp_save_rate':    row[f'{prefix}bp_save_rate']    / 100,
        'bp_save_faced':   max(1, int(row[f'{prefix}bp_save_faced'])),
        'bp_convert_rate': row[f'{prefix}bp_convert_rate'] / 100,
        'bp_convert_opps': max(1, int(row[f'{prefix}bp_convert_opps'])),
    }


results = []  # list of (match_id, p1_win_prob, outcome, year)
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='Simulating'):
    try:
        s1 = make_stats(row, 'p1_')
        s2 = make_stats(row, 'p2_')
        p1 = Player('p1', s1, prior_strength=PRIOR_STRENGTH, lam=LAM)
        p2 = Player('p2', s2, prior_strength=PRIOR_STRENGTH, lam=LAM)
        best_of = int(row['number_of_sets']) if row['number_of_sets'] else 3
        p1_prob = run_simulation(p1, p2, N=N_SIMS, best_of=best_of)
        outcome = 1 if row['winner_player_id'] == row['p1_id'] else 0
        results.append({
            'match_id':   row['match_id'],
            'event_year': row['event_year'],
            'tournament': row['tournament_name'],
            'p1_id':      row['p1_id'],
            'p2_id':      row['p2_id'],
            'p1_prob':    p1_prob,
            'outcome':    outcome,
        })
    except Exception as e:
        skipped += 1

results_df = pd.DataFrame(results)
print(f'Simulated: {len(results_df):,}  |  Skipped: {skipped}')

Simulating: 100%|██████████| 10212/10212 [2:49:42<00:00,  1.00it/s] 

Simulated: 10,212  |  Skipped: 0


In [6]:
# ── Brier Score ───────────────────────────────────────────────────────────────

results_df['brier'] = (results_df['p1_prob'] - results_df['outcome']) ** 2

overall = results_df['brier'].mean()
print(f'Overall Brier score : {overall:.4f}  (random baseline = 0.2500)')
print(f'Skill score         : {1 - overall/0.25:.4f}  (0=random, 1=perfect)')
print()

print('By year:')
by_year = results_df.groupby('event_year')['brier'].agg(['mean','count'])
by_year.columns = ['brier_score', 'n_matches']
print(by_year.to_string())

Overall Brier score : 0.2196  (random baseline = 0.2500)
Skill score         : 0.1215  (0=random, 1=perfect)

By year:
            brier_score  n_matches
event_year                        
2023           0.216490       2347
2024           0.217491       2959
2025           0.225343       3563
2026           0.214647       1343


In [7]:
# ── Calibration check ─────────────────────────────────────────────────────────
# Bucket predictions into deciles and check actual win rates

results_df['bucket'] = pd.cut(results_df['p1_prob'], bins=10, precision=1)
cal = results_df.groupby('bucket', observed=True).agg(
    n=('outcome','count'),
    mean_pred=('p1_prob','mean'),
    actual_win_rate=('outcome','mean')
).reset_index()
print('Calibration (predicted vs actual win rate):')
print(cal[['bucket','n','mean_pred','actual_win_rate']].to_string(index=False))

Calibration (predicted vs actual win rate):
       bucket    n  mean_pred  actual_win_rate
(-0.001, 0.1]  179   0.016540         0.782123
   (0.1, 0.2]  102   0.154245         0.617647
   (0.2, 0.3]  278   0.257873         0.697842
   (0.3, 0.4]  919   0.358992         0.842220
   (0.4, 0.5] 2225   0.455726         0.935281
   (0.5, 0.6] 2936   0.549313         0.955722
   (0.6, 0.7] 1924   0.644292         0.962058
   (0.7, 0.8]  803   0.738913         0.975093
   (0.8, 0.9]  252   0.847925         0.952381
   (0.9, 1.0]  594   0.990565         0.983165


In [8]:
# ── Save results ──────────────────────────────────────────────────────────────
results_df.to_csv('brier_backtest_results.csv', index=False)
print('Saved to brier_backtest_results.csv')

Saved to brier_backtest_results.csv
